# Fabric IQ and Foundry IQ agents — companion notebook

Companion to the end-to-end deck **Microsoft Foundry Fabric IQ and Foundry IQ Agents** 
One scenario runs through every step: the operations centre of the fictional **Valmont Winter Games**, built from the kit sample `samples/games-operations` (venues, sessions, assets, live telemetry, inspection findings and 51 PDF documents).

Steps 0 to 4 cover the knowledge layer: the two Fabric MCP endpoints, a document indexing pipeline in Azure AI
Search, and Foundry IQ knowledge sources side by side. Steps 5 and 6 build three agents on that layer and evaluate
them. Steps 7 to 11 (permissions, Work IQ, publishing, Copilot Studio, API Management) follow in later updates of
this notebook.

**Before you start (Fabric side)** — built with the sample, in this order
- `GamesLakehouse` loaded by `samples/games-operations/fabric/01_load_lakehouse.ipynb`, with the PDFs under
  **Files** → `games_ops/output/documents` (four subfolders: manuals, inspections, transport, policies)
- `GamesEventhouse` (`fabric/02_eventhouse_setup.kql`), the semantic model **Games Operations Model**
  (`fabric/03_build_semantic_model.ipynb`), the findings table (`fabric/04_extract_findings.ipynb`) and the
  ontology `GamesOperationsOntology` finished with the three prompts in `ontology/ontology-agent-prompts.md`
- A Fabric capacity F2 or higher; the workspace is not *My workspace*; the lakehouse has no sensitivity label
  (as per official documentation, the OneLake indexer can't read labelled Fabric items)

**Before you start (Azure side)**
- An Azure AI Search service, **Basic tier or higher**, semantic ranker enabled, with a **system-assigned managed
  identity**, in the same Microsoft Entra tenant as the Fabric workspace
- The Search service identity has **Contributor** on the Fabric workspace (workspace **Manage access**)
- The Search service identity has **Cognitive Services User** and **Cognitive Services OpenAI User** on the Foundry
  resource (content extraction billing, embeddings, knowledge base model)
- The Foundry resource is in a region that supports Azure Content Understanding (used by step 3)
- A Foundry project with a chat model deployment (`FOUNDRY_MODEL_NAME`, a model supported by knowledge bases, for
  example `gpt-5-mini` or `gpt-4.1-mini`) and an embedding deployment (`FOUNDRY_EMBEDDING_NAME`,
  `text-embedding-3-small` or `text-embedding-3-large`)
- Your own account has **Search Service Contributor** and **Search Index Data Contributor** on the search service,
  read access to the Fabric workspace and ontology, and is signed in with `az login` (the ontology steps use your
  sign-in; as per official documentation, service principals can't call the ontology MCP endpoint or use Fabric
  IQ MCP)
- For steps 5 and 6: `AZURE_SUBSCRIPTION_ID` and `AZURE_RESOURCE_GROUP` (the resource group of the Foundry account)
  in `.env`; your account has **Foundry Project Manager** on the project (project connections) and may assign roles
  at project scope (**Owner** or **User Access Administrator**), or an administrator assigns **Foundry User** to the
  two agent identities after step 5
- The project's system-assigned managed identity has **Search Index Data Reader** on the search service (the agent
  knowledge base connection)
- The ontology is **published** in Fabric (as per official documentation, unpublished items aren't reachable through
  Fabric IQ and return 404), and every user who calls the agents has a Fabric licence and read access to it
- For step 5b: a project region and subscription enabled for the Microsoft Copilot harness preview, and a
  `FOUNDRY_MODEL_NAME` that supports reasoning and tool calling
- For the trace query in step 6: Application Insights connected to the project and its resource ID in
  `APP_INSIGHTS_RESOURCE_ID` (optional; the Foundry portal shows the same traces)
- The generated sample output in `samples/games-operations/output` (step 0 regenerates the register and questions
  if it is missing)

| Step | What it shows | Status | Deck section |
|---|---|---|---|
| 0 | Settings, clients and the sample's evaluation questions | GA | — |
| 1 | The ontology as an MCP server: its tools and one question | preview | 2 |
| 2 | Fabric IQ MCP over the semantic model: its tools and one discovery call | GA | 2 |
| 3 | Document indexing pipeline: OneLake data source, skillset, index, indexer, chunk inspection | GA | 4 |
| 4 | Knowledge sources side by side: custom index, indexed OneLake, Fabric ontology; one knowledge base | preview | 3 |
| 5 | Three agent patterns on one knowledge layer: prompt agent, prompt agent with the Copilot harness, hosted Agent Framework agent | preview | 5, 6 |
| 6 | Evaluations (agent targets and a response dataset) and traces | preview | 5 |
| — | Cleanup of the resources created in steps 3 to 6 | GA | — |

Chargeable cells start with `# CHARGEABLE:` and destructive cells with `# DESTRUCTIVE:`. Azure calls run only when
every required setting is present (`LIVE`).

Lines marked `# verify` use a shape that was documented at the time of writing but not executed here — check them against the Learn page linked in the cell before you present.

## 0 · Settings and clients

One `.env` at the repository root, shared with the other kit notebooks. This notebook adds
`AZURE_SEARCH_ENDPOINT` and the Fabric item IDs (`FABRIC_*`); the IDs are in the item URLs in the Fabric portal,
for example `https://app.fabric.microsoft.com/groups/<workspace-id>/ontologies/<ontology-id>`.
Authentication is Microsoft Entra ID through `DefaultAzureCredential`; no keys. Install the shared
dependencies once from the repository root with `python3 -m pip install -r requirements.txt` (see Local Setup in
the kit README); it includes the sample's requirements.

Run with the repository root as the working directory. The first cell only loads settings and prints the exact
`.env` path and any missing names; it makes no Azure calls. Save `.env` and rerun that cell after edits; no kernel
restart is needed. File values override existing environment variables. A missing `FABRIC_WORKSPACE_ID` must be
added to `.env` explicitly: the lakehouse and ontology IDs do not supply the workspace ID.
Each Fabric ID must be just a GUID. For `FABRIC_ONTOLOGY_ID`, copy only the GUID after `/ontologies/`,
not any following `/entity/<id>` path or URL query parameters.

In [3]:
import os
from pathlib import Path
from uuid import UUID
from dotenv import load_dotenv

ENV_FILE = Path(".env").resolve()
if not ENV_FILE.is_file():
    raise FileNotFoundError(f"No .env at {ENV_FILE}. Run from the repository root and save your .env there.")
load_dotenv(ENV_FILE, override=True)
print(f"Settings file: {ENV_FILE}")

def env(name, default=""):
    return (os.getenv(name) or default).strip()

FOUNDRY_PROJECT_ENDPOINT = env("FOUNDRY_PROJECT_ENDPOINT")
FOUNDRY_MODEL_NAME = env("FOUNDRY_MODEL_NAME")
FOUNDRY_EMBEDDING_NAME = env("FOUNDRY_EMBEDDING_NAME")
AZURE_SEARCH_ENDPOINT = env("AZURE_SEARCH_ENDPOINT")
FABRIC_WORKSPACE_ID = env("FABRIC_WORKSPACE_ID")
FABRIC_LAKEHOUSE_ID = env("FABRIC_LAKEHOUSE_ID")
FABRIC_ONTOLOGY_ID = env("FABRIC_ONTOLOGY_ID")
FABRIC_DOCS_FOLDER = env("FABRIC_DOCS_FOLDER", "games_ops/output/documents")

required = ("FOUNDRY_PROJECT_ENDPOINT", "FOUNDRY_MODEL_NAME", "FOUNDRY_EMBEDDING_NAME",
            "AZURE_SEARCH_ENDPOINT", "FABRIC_WORKSPACE_ID", "FABRIC_LAKEHOUSE_ID", "FABRIC_ONTOLOGY_ID")
missing = [name for name in required if not env(name) or "<" in env(name)]
invalid_ids = []
for name in ("FABRIC_WORKSPACE_ID", "FABRIC_LAKEHOUSE_ID", "FABRIC_ONTOLOGY_ID"):
    if name not in missing:
        try:
            UUID(env(name))
        except ValueError:
            invalid_ids.append(name)
LIVE = not missing and not invalid_ids
print("Required settings missing:", ", ".join(missing) if missing else "none")
if invalid_ids:
    print("Invalid Fabric IDs:", ", ".join(invalid_ids), "— use GUIDs only, without /entity/... or URL parameters.")

Settings file: /workspaces/Microsoft-Foundry-What-s-New-and-Deep-Dive-Kit/.env
Required settings missing: none


### Clients and model metadata

Run this cell after checking the settings above. Model lookups use your Azure sign-in and only run when
`LIVE` is true. Keep model names separate from deployment names: Azure AI Search needs both.

In [4]:
import sys, json, time, re, subprocess
from urllib.parse import urlparse
import pandas as pd
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

credential = DefaultAzureCredential()            # your az login identity; the Fabric steps need a user, not a service principal
ACCOUNT = (urlparse(FOUNDRY_PROJECT_ENDPOINT).hostname or "account").split(".")[0]
FOUNDRY_OPENAI_URL = f"https://{ACCOUNT}.openai.azure.com"          # embeddings, vectorizer and knowledge base model
FOUNDRY_SERVICES_URL = f"https://{ACCOUNT}.services.ai.azure.com"   # resource that bills content extraction in step 3

CHAT_MODEL, EMBED_MODEL = FOUNDRY_MODEL_NAME, FOUNDRY_EMBEDDING_NAME
if LIVE:
    project = AIProjectClient(endpoint=FOUNDRY_PROJECT_ENDPOINT, credential=credential)
    for var in ("CHAT_MODEL", "EMBED_MODEL"):
        deployment = globals()[var]
        try:   # the search service needs the model name as well as the deployment name
            globals()[var] = project.deployments.get(deployment).model_name   # verify https://learn.microsoft.com/en-us/python/api/azure-ai-projects/
        except Exception as e:
            print(f"Model name for {deployment} not read ({type(e).__name__}); using the deployment name")
EMBED_DIMS = {"text-embedding-3-large": 3072, "text-embedding-3-small": 1536, "text-embedding-ada-002": 1536}.get(EMBED_MODEL, 1536)
print(f"chat model {CHAT_MODEL} · embedding model {EMBED_MODEL} ({EMBED_DIMS} dimensions)")

chat model gpt-6.1-sol · embedding model text-embedding-3-large (3072 dimensions)


The sample's document register and evaluation questions come from the generated output. The register adds the
document type, venue, version and status to every chunk in step 3; the questions and expected answers drive the
comparisons in step 4 and the evaluations in step 6. Generation is deterministic, so the IDs match the data
loaded into Fabric.

In [7]:
SAMPLE = Path("samples/games-operations")
SAMPLE_OUT = SAMPLE / "output"
if not (SAMPLE_OUT / "lakehouse" / "document_register.csv").exists():
    print("Sample output not found; generating the register and questions (no PDFs) ...")
    subprocess.run([sys.executable, str(SAMPLE / "generator" / "generate.py"), "--out", str(SAMPLE_OUT), "--no-docs"], check=True)

register = pd.read_csv(SAMPLE_OUT / "lakehouse" / "document_register.csv", dtype=str, keep_default_na=False)
questions = pd.read_csv(SAMPLE_OUT / "eval" / "questions.csv", dtype=str, keep_default_na=False).set_index("id")
print(f"{len(register)} documents in the register, {len(questions)} evaluation questions")
display(register.groupby(["document_type", "document_status"]).size().rename("documents").reset_index())

# Names of everything this notebook creates in Azure AI Search
INDEX, DATASOURCE, SKILLSET, INDEXER = "games-docs", "games-docs-onelake", "games-docs-skillset", "games-docs-indexer"
KS_DOCS, KS_ONELAKE, KS_ONTOLOGY, KB = "games-docs-ks", "games-onelake-ks", "games-ontology-ks", "games-kb"
SEARCH_API_VERSION = "2026-08-01-preview"   # knowledge bases with answer synthesis, reasoning effort and the Fabric ontology source are preview

51 documents in the register, 18 evaluation questions


,document_type,document_status,documents
0,Games-wide policy,Current,3
1,Inspection report,Current,34
2,Transport plan,Current,2
3,Venue operations manual,Current,10
4,Venue operations manual,Superseded,2


## 1 · Ontology as an MCP server (preview)  *(deck section 2)*

Fabric exposes each ontology item as a Model Context Protocol (MCP) endpoint. An agent connected to it can list
the entity types and ask questions in business terms; the ontology resolves them against its bindings
(lakehouse, eventhouse, semantic model). The call runs with your own sign-in: as per official documentation, a
service principal can't call the ontology MCP endpoint yet, and the capacity must be F2 or higher.

Source: https://learn.microsoft.com/en-us/fabric/iq/ontology/how-to-use-ontology-mcp-server

Use the official Python MCP SDK directly: 
- `ClientSession.list_tools()` discovers the server's tools and
- `ClientSession.call_tool()` invokes them. 

### Connect and discover the ontology's tools

Check the configured item before opening the MCP session.

In [9]:
import httpx
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

FABRIC_SCOPE = "https://api.fabric.microsoft.com/.default"
ONTOLOGY_MCP_URL = (f"https://api.fabric.microsoft.com/v1/mcp/dataPlane/workspaces/{FABRIC_WORKSPACE_ID}"
                    f"/items/{FABRIC_ONTOLOGY_ID}/ontologyEndpoint")
ontology_tools = []
if LIVE:
    headers = {"Authorization": f"Bearer {credential.get_token(FABRIC_SCOPE).token}",
               "x-ms-fabric-skill": "fabriciq-ontology-cli"}
    item_url = f"https://api.fabric.microsoft.com/v1/workspaces/{FABRIC_WORKSPACE_ID}/items/{FABRIC_ONTOLOGY_ID}"
    async with httpx.AsyncClient(headers=headers, timeout=30) as client:
        response = await client.get(item_url)
        if response.status_code == 404:
            raise ValueError("Ontology item not found. Check FABRIC_WORKSPACE_ID and FABRIC_ONTOLOGY_ID in .env.")
        response.raise_for_status()
        item = response.json()
    if item["type"] != "Ontology":
        raise ValueError(f"FABRIC_ONTOLOGY_ID points to a {item['type']}, not an Ontology.")
    print(f"Ontology: {item['displayName']}")
    async with streamablehttp_client(ONTOLOGY_MCP_URL, headers=headers) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            ontology_tools = (await session.list_tools()).tools
    display(pd.DataFrame([t.model_dump(include={"name", "description", "inputSchema"}) for t in ontology_tools]))
else:
    print("Settings missing — step 1 skipped")

Ontology: GamesOperationsOntology


,name,description,inputSchema
0,ask_ontology,Answers day-to-day business questions using an...,"{'type': 'object', 'properties': {'request': {..."
1,list_ontology_rules,"Lists the business rule definitions, including...","{'type': 'object', 'properties': {'entityNames..."
2,list_ontology_entities,"Lists the entities defined in the ontology, in...","{'type': 'object', 'properties': {'entityName'..."


One question through the ontology: evaluation question Q12 depends on the `ServiceOverdue` rule, so the answer shows whether the rule travels with the ontology. Call the server's `ask_ontology` tool with its published `request` 

In [10]:
if LIVE:
    if "ask_ontology" not in {tool.name for tool in ontology_tools}:
        raise RuntimeError("ask_ontology was not discovered. Run the ontology connection cell first.")
    q = questions.loc["Q12"]
    print(f"Tool ask_ontology · question: {q.question}\n")
    headers = {"Authorization": f"Bearer {credential.get_token(FABRIC_SCOPE).token}",
               "x-ms-fabric-skill": "fabriciq-ontology-cli"}
    async with streamablehttp_client(ONTOLOGY_MCP_URL, headers=headers) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool("ask_ontology", {"request": q.question})
    answer = "\n".join(content.text for content in result.content if content.type == "text")
    if result.isError:
        raise RuntimeError(f"ask_ontology failed: {answer}")
    print(answer[:3000])
    print("\nExpected:", q.expected_answer)

Tool ask_ontology · question: Which Critical assets are overdue for service (more than 30 days since the last service)?



Using **last_service_date older than 30 days before today**; I found **1 critical asset** overdue for service.

- **Asset ID:** V08-AS-020  
  **Name:** Safety netting section 1  
  **Last service date:** 2026-10-07

If you want, I can also list the **days overdue** for each asset.

Expected: V08-AS-020 (Safety netting section, Summit Alpine Centre, last service 2026-10-07)


## 2 · Fabric IQ MCP over the semantic model  *(deck section 2)*

Fabric IQ MCP is the generally available, tenant-wide MCP server for Power BI semantic models: an agent discovers
models and reports, reads a model's schema and runs DAX queries, with the user's permissions (row-level security
applies). As per official documentation it doesn't reach ontologies or data agents yet, so it complements step 1
rather than replacing it. The `X-Variants` header pins the tool contract to version 1.

| Endpoint | What an agent reaches | Status |
|---|---|---|
| Ontology MCP (step 1) | One ontology: entity types, relationships, rules, live readings | Preview |
| Fabric IQ MCP (this step) | Every semantic model and report the user can read; DAX queries | GA |

Source: https://learn.microsoft.com/en-us/fabric/iq/connectors/fabric-iq-mcp

This cell uses the same SDK directly and can run after step 0 without running step 1. It discovers the
published tools, then calls `DiscoverArtifacts` with its `searchQuery` argument in the same MCP session.

In [11]:
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

POWERBI_SCOPE = "https://analysis.windows.net/powerbi/api/.default"
FABRIC_IQ_MCP_URL = "https://fabriciq.svc.cloud.microsoft/v1/mcp/fabriciq"   # private link tenants: https://api.fabric.microsoft.com/v1/mcp/fabriciq
fabric_iq_tools = []
if LIVE:
    headers = {"Authorization": f"Bearer {credential.get_token(POWERBI_SCOPE).token}",
               "X-Variants": "Fabric.Routing.FabricIQ.V1",
               "x-ms-fabric-skill": "fabriciq-ontology-cli"}
    async with streamablehttp_client(FABRIC_IQ_MCP_URL, headers=headers) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            fabric_iq_tools = (await session.list_tools()).tools
            display(pd.DataFrame([t.model_dump(include={"name", "description", "inputSchema"}) for t in fabric_iq_tools]))
            if "DiscoverArtifacts" not in {tool.name for tool in fabric_iq_tools}:
                raise RuntimeError("DiscoverArtifacts was not exposed by the Fabric IQ MCP server.")
            result = await session.call_tool("DiscoverArtifacts", {"searchQuery": "Games Operations Model"})
    discovery = "\n".join(content.text for content in result.content if content.type == "text")
    if result.isError:
        raise RuntimeError(f"DiscoverArtifacts failed: {discovery}")
    print(discovery[:2000])
else:
    print("Settings missing — step 2 skipped")

,name,description,inputSchema
0,ValueSearch,Resolves entity names and filter values to exa...,"{'type': 'object', 'properties': {'artifactId'..."
1,ResolveFabricItem,Resolves a Fabric or Power BI item to its cano...,"{'type': 'object', 'properties': {'fabricItemI..."
2,DiscoverArtifacts,Searches for Power BI artifacts (Semantic Mode...,"{'type': 'object', 'properties': {'searchQuery..."
3,GetReportMetadata,"Retrieves the schema of a Power BI report, inc...","{'type': 'object', 'properties': {'reportObjec..."
4,ExecuteQuery,Executes a DAX query and return the results. T...,"{'type': 'object', 'properties': {'artifactId'..."
5,GetSemanticModelSchema,This tool retrieves comprehensive metadata for...,"{'type': 'object', 'properties': {'artifactId'..."


{"searchQuery":"Games Operations Model","totalResults":5,"artifacts":[{"Name":"Games Operations Model","ArtifactId":"4d6580a5-a5c1-46de-90ce-3473ceae086c","ArtifactType":"Semantic Model","DirectLink":"https://app.powerbi.com/datahub/datasets/4d6580a5-a5c1-46de-90ce-3473ceae086c","WorkspaceName":"IOC_Demo","Description":"The model tracks Olympic athlete, event, venue, and medal data with key measures on medals, ticket sales, attendance, and venue session completions.","IconUrl":"https://res.cdn.office.net/files/fabric-cdn-prod_20260917.001/assets/item-types/16/pbisemmodel.svg","SensitivityLabelId":"","lastOpenedUtc":"2026-11-10T12:59:45.79Z","relevance":"SomewhatRelevant","matchedSignals":["Recents","ArtifactType"]},{"Name":"Fabric Capacity Metrics","ArtifactId":"60015ad4-d8d9-4f9e-8fbd-0ee08415825d","ArtifactType":"Report","DirectLink":"https://app.powerbi.com/groups/9a953475-15aa-48e3-8891-f0aae35cdc31/reports/60015ad4-d8d9-4f9e-8fbd-0ee08415825d","WorkspaceName":"Microsoft Fabric Cap

## 3 · Document indexing pipeline  *(deck section 4)*

The ontology answers *which, how many, related to what*. The documents answer *what exactly does the manual or
the inspector say*. This step indexes the 51 PDFs straight from the lakehouse **Files** area, with no copy:

| Object | Role in the pipeline |
|---|---|
| Data source (`onelake`) | Points at the workspace, the lakehouse and the documents folder |
| Skillset | Extracts text and layout, cuts it into chunks, adds an embedding to each chunk |
| Index projections | Write one search document per chunk, linked to its parent file |
| Index | Text, vector and filter fields; semantic configuration and vectorizer for agentic retrieval |
| Indexer | Runs the pipeline and tracks changed files on later runs |

Extraction uses the **Azure Content Understanding skill**, which as per official documentation replaces the
Document Layout skill for new skillsets: it returns Markdown chunks with page numbers, so no separate split
skill is needed. It reads the file bytes (`file_data`); verify that your OneLake indexer supplies them. The
alternative `CHUNKER = "builtIn"` uses the indexer's PDF text extraction plus Text Split (no page citations).
Switching chunkers is explicit: set `RESET_INDEXER = True` and rerun configuration, run, status and enrichment.
Reset reprocesses all PDFs and bills extraction/embeddings again; there is no automatic second run.

Use the Azure AI Search SDK models and client methods directly: define the index and skills, start the indexer,
then inspect its status. The configuration cells make no service calls; only the run cell starts billable work.

Sources: https://learn.microsoft.com/en-us/azure/search/search-how-to-index-onelake-files ·
https://learn.microsoft.com/en-us/azure/search/cognitive-search-skill-content-understanding ·
https://learn.microsoft.com/en-us/azure/search/search-how-to-semantic-chunking-content-understanding ·
https://learn.microsoft.com/en-us/azure/search/agentic-retrieval-how-to-create-index

In [16]:
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient, SearchIndexerClient, models

if LIVE:
    index_client = SearchIndexClient(endpoint=AZURE_SEARCH_ENDPOINT, credential=credential, api_version=SEARCH_API_VERSION)
    indexer_client = SearchIndexerClient(endpoint=AZURE_SEARCH_ENDPOINT, credential=credential, api_version=SEARCH_API_VERSION)
    search_client = SearchClient(endpoint=AZURE_SEARCH_ENDPOINT, index_name=INDEX, credential=credential, api_version=SEARCH_API_VERSION)

index_definition = models.SearchIndex(
    name=INDEX,
    fields=[
        models.SearchableField(name="chunk_id", key=True, filterable=True, sortable=True, analyzer_name="keyword"),
        models.SearchableField(name="chunk"),
        models.SearchField(name="text_vector", type="Collection(Edm.Single)", searchable=True, hidden=True,
                           stored=False, vector_search_dimensions=EMBED_DIMS, vector_search_profile_name="games-hnsw-profile"),
        models.SearchableField(name="file_name", filterable=True, facetable=True),
        models.SearchableField(name="title", filterable=True, facetable=True),
        # Match the original REST defaults; these attributes cannot change on an existing field.
        models.SimpleField(name="page_from", type="Edm.Int32", filterable=True, sortable=True, facetable=True),
        models.SimpleField(name="page_to", type="Edm.Int32", filterable=True, sortable=True, facetable=True),
        *[models.SimpleField(name=name, type="Edm.String", filterable=True, facetable=True)
          for name in ("parent_id", "document_id", "document_type", "venue_id", "version", "document_status", "sensitivity")],
    ],
    vector_search=models.VectorSearch(
        algorithms=[models.HnswAlgorithmConfiguration(name="games-hnsw")],
        profiles=[models.VectorSearchProfile(name="games-hnsw-profile", algorithm_configuration_name="games-hnsw",
                                             vectorizer_name="games-vectorizer")],
        vectorizers=[models.AzureOpenAIVectorizer(vectorizer_name="games-vectorizer",
            parameters=models.AzureOpenAIVectorizerParameters(resource_url=FOUNDRY_OPENAI_URL,
                deployment_name=FOUNDRY_EMBEDDING_NAME, model_name=EMBED_MODEL))]),
    semantic_search=models.SemanticSearch(default_configuration_name="games-semantic", configurations=[
        models.SemanticConfiguration(name="games-semantic", prioritized_fields=models.SemanticPrioritizedFields(
            title_field=models.SemanticField(field_name="title"),
            content_fields=[models.SemanticField(field_name="chunk")],
            keywords_fields=[models.SemanticField(field_name="document_type"), models.SemanticField(field_name="venue_id")]))]),
)

### Configure the data source, skills and indexer

The search service reads OneLake with its own managed identity (Contributor on
the workspace). Content extraction is billed to the Foundry resource attached to the skillset by identity, with no
key; as per official documentation, the Content Understanding skill has no free daily allowance.

In [17]:
CHUNKER = "contentUnderstanding"   # or "builtIn": PDF text extraction plus Text Split
RESET_INDEXER = False             # True when switching CHUNKER: reprocess all PDFs (billable)

datasource = models.SearchIndexerDataSourceConnection(
    name=DATASOURCE, type="onelake", connection_string=f"ResourceId={FABRIC_WORKSPACE_ID}",
    container=models.SearchIndexerDataContainer(name=FABRIC_LAKEHOUSE_ID, query=FABRIC_DOCS_FOLDER))

if CHUNKER == "contentUnderstanding":
    context = "/document/text_sections/*"
    text_source = f"{context}/content"
    chunking_skill = models.ContentUnderstandingSkill(
        name="extract-and-chunk", context="/document", extraction_options=["locationMetadata"],
        chunking_properties=models.ContentUnderstandingSkillChunkingProperties(unit="characters", maximum_length=2000, overlap_length=200),
        inputs=[models.InputFieldMappingEntry(name="file_data", source="/document/file_data")],
        outputs=[models.OutputFieldMappingEntry(name="text_sections", target_name="text_sections")])
    page_mappings = [models.InputFieldMappingEntry(name="page_from", source=f"{context}/locationMetadata/pageNumberFrom"),
                     models.InputFieldMappingEntry(name="page_to", source=f"{context}/locationMetadata/pageNumberTo")]
elif CHUNKER == "builtIn":
    context = text_source = "/document/pages/*"
    chunking_skill = models.SplitSkill(
        name="chunk", context="/document", text_split_mode="pages", maximum_page_length=2000,
        page_overlap_length=200, default_language_code="en",
        inputs=[models.InputFieldMappingEntry(name="text", source="/document/content")],
        outputs=[models.OutputFieldMappingEntry(name="textItems", target_name="pages")])
    page_mappings = []
else:
    raise ValueError("CHUNKER must be 'contentUnderstanding' or 'builtIn'.")

embedding_skill = models.AzureOpenAIEmbeddingSkill(
    name="embed", context=context, resource_url=FOUNDRY_OPENAI_URL,
    deployment_name=FOUNDRY_EMBEDDING_NAME, model_name=EMBED_MODEL, dimensions=EMBED_DIMS,
    inputs=[models.InputFieldMappingEntry(name="text", source=text_source)],
    outputs=[models.OutputFieldMappingEntry(name="embedding", target_name="text_vector")])
skillset = models.SearchIndexerSkillset(
    name=SKILLSET, skills=[chunking_skill, embedding_skill],
    cognitive_services_account=models.AIServicesAccountIdentity(subdomain_url=FOUNDRY_SERVICES_URL),
    index_projection=models.SearchIndexerIndexProjection(selectors=[models.SearchIndexerIndexProjectionSelector(
        target_index_name=INDEX, parent_key_field_name="parent_id", source_context=context,
        mappings=[models.InputFieldMappingEntry(name="chunk", source=text_source), *page_mappings,
                  models.InputFieldMappingEntry(name="text_vector", source=f"{context}/text_vector"),
                  models.InputFieldMappingEntry(name="file_name", source="/document/metadata_storage_name")])],
        parameters=models.SearchIndexerIndexProjectionsParameters(projection_mode="skipIndexingParentDocuments")))
indexer = models.SearchIndexer(
    name=INDEXER, data_source_name=DATASOURCE, target_index_name=INDEX, skillset_name=SKILLSET,
    parameters=models.IndexingParameters(configuration=models.IndexingParametersConfiguration(
        data_to_extract="contentAndMetadata", parsing_mode="default", indexed_file_name_extensions=".pdf",
        allow_skillset_to_read_file_data=CHUNKER == "contentUnderstanding")),
    field_mappings=[], output_field_mappings=[])

### Create or update the pipeline and run once

A new indexer starts automatically. An existing indexer needs an explicit run; reset it only when you want to
reprocess unchanged PDFs (for example after switching chunkers). Run this cell once, then use the status cell.

In [18]:
# CHARGEABLE: content extraction per page and embeddings per chunk (51 short PDFs).
from azure.core.exceptions import ResourceNotFoundError

if LIVE:
    try:
        status = indexer_client.get_indexer_status(INDEXER)
    except ResourceNotFoundError:
        status = None
    if status and status.last_result and status.last_result.status == "inProgress":
        raise RuntimeError("The indexer is already running. Wait for it to finish before changing the pipeline.")
    previous_end = status.last_result.end_time if status and status.last_result else None
    index_client.create_or_update_index(index_definition)
    indexer_client.create_or_update_data_source_connection(datasource)
    indexer_client.create_or_update_skillset(skillset)
    indexer_client.create_or_update_indexer(indexer)
    if status is not None:
        if RESET_INDEXER:
            indexer_client.reset_indexer(INDEXER)
        indexer_client.run_indexer(INDEXER)
    print(f"Started {INDEXER} with {CHUNKER}. Run the status cell next.")

Started games-docs-indexer with contentUnderstanding. Run the status cell next.


### Wait for completion and inspect the result

The SDK starts indexers asynchronously, so this short loop polls for up to 20 minutes. It ignores the previous
run's result and raises on failure or timeout. This cell never starts or resets an indexer.

In [12]:
if LIVE:
    deadline = time.monotonic() + 1200
    while time.monotonic() < deadline:
        last = indexer_client.get_indexer_status(INDEXER).last_result
        if last and last.status != "inProgress" and last.end_time and last.end_time != previous_end:
            break
        print("Indexer:", last.status if last else "starting", end="\r")
        time.sleep(15)
    else:
        raise TimeoutError("Indexer not finished after 20 minutes; it may still be running. Rerun this status cell.")
    print(f"\n{CHUNKER}: {last.status}, files {last.item_count}, failed {last.failed_item_count}")
    for warning in last.warnings:
        print("Warning:", warning.message)
    for error in last.errors:
        print("Error:", error.error_message)
    if last.status != "success" or last.failed_item_count:
        raise RuntimeError(f"Indexer failed: {last.error_message or last.status}. Review the errors above.")
    chunks = search_client.get_document_count()
    print("Chunks in index:", chunks)
    if chunks == 0:
        raise RuntimeError("No searchable chunks yet. Rerun this status cell after search refresh. If still empty, "
                           "review the warnings; for the Text Split alternative set CHUNKER='builtIn' and "
                           "RESET_INDEXER=True, then rerun configuration, run and status.")


contentUnderstanding: IndexerExecutionStatus.SUCCESS, files 0, failed 0
Chunks in index: 63


Add document-version information to the searchable text, so the agent can distinguish current guidance from outdated guidance.
The indexing pipeline splits each PDF into searchable text chunks and creates their embeddings.
This additional cell matches each chunk’s filename to the catalogue.

In [13]:
if LIVE:
    reg = register.set_index("file_name")
    cols = ["document_id", "title", "document_type", "venue_id", "version", "document_status", "sensitivity"]
    rows = list(search_client.search(search_text="*", select=["chunk_id", "file_name"], top=1000))
    updates = [{"chunk_id": r["chunk_id"], **reg.loc[r["file_name"], cols].to_dict()} for r in rows if r["file_name"] in reg.index]
    for i in range(0, len(updates), 500):
        search_client.merge_documents(documents=updates[i:i + 500])
    print(f"Register fields merged into {len(updates)} of {len(rows)} chunks")

Register fields merged into 63 of 63 chunks


Chunk inspection: chunks per document type and status, then the chunks that hold finding F-0034-1 and the two
versions of the Ridge Ski Jumping Hill wind limit.

In [19]:
if LIVE:
    time.sleep(2)
    facets = search_client.search(search_text="*", facets=["document_type", "document_status"], top=0).get_facets()
    for name, values in facets.items():
        print(name, {v["value"]: v["count"] for v in values})

    for query, flt in [("F-0034-1 compressor 2 bearing", "document_type eq 'Inspection report'"),
                       ("wind gust limit consecutive minutes", "venue_id eq 'V07'")]:
        print(query)
        hits = search_client.search(search_text=query, filter=flt, top=3,
                                    select=["document_id", "version", "document_status", "page_from", "chunk"])
        display(pd.DataFrame([{"document": h.get("document_id"), "version": h.get("version"),
                               "status": h.get("document_status"), "page": h.get("page_from"),
                               "chunk": (h.get("chunk") or "")[:300]} for h in hits]))

document_type {'Inspection report': 34, 'Venue operations manual': 24, 'Games-wide policy': 3, 'Transport plan': 2}
document_status {'Current': 59, 'Superseded': 4}
F-0034-1 compressor 2 bearing


,document,version,status,page,chunk
0,DOC-IR-0034,1.0,Current,1,Venue Inspection Report: Summit Sliding Centre...
1,DOC-IR-0020,1.0,Current,1,Venue Inspection Report: Summit Sliding Centre...
2,DOC-IR-0031,1.0,Current,1,Venue Inspection Report: Valmont Ice Arena<tab...


wind gust limit consecutive minutes


,document,version,status,page,chunk
0,DOC-MAN-V07-10,1.0,Superseded,1,Ridge Ski Jumping Hill Operations Manual<table...
1,DOC-MAN-V07-21,2.1,Current,1,Ridge Ski Jumping Hill Operations Manual<table...
2,DOC-MAN-V07-21,2.1,Current,1,re for five minutes or more. When the limit is...


## 4 · Knowledge sources side by side (preview)  *(deck section 3)*

Foundry IQ is the knowledge layer of Azure AI Search: **knowledge sources** describe where content lives, and a
**knowledge base** plans the queries, retrieves from its sources in parallel and returns grounding data or a
synthesised answer with citations. This step registers three sources over the same scenario and puts them in
one knowledge base:

| Knowledge source | Content | Who builds the pipeline |
|---|---|---|
| `games-docs-ks` (search index) | The step 3 index: chunks with register fields | Development team (step 3) |
| `games-onelake-ks` (indexed OneLake) | The same PDF folder | The knowledge source generates data source, skillset, index and indexer |
| `games-ontology-ks` (Fabric ontology) | The ontology from step 1 | Nothing to index: queried live with the user's identity |

The comparison asks the same questions one source at a time (a retrieve request can name the sources to query),
with the same model, reasoning effort and output mode, and reports answer, citations, tokens and time.
The Fabric ontology source needs the user's token on every request (`x-ms-query-source-authorization`) and
doesn't support the *minimal* reasoning effort.

Sources: https://learn.microsoft.com/en-us/azure/search/agentic-knowledge-source-how-to-search-index ·
https://learn.microsoft.com/en-us/azure/search/agentic-knowledge-source-how-to-onelake ·
https://learn.microsoft.com/en-us/azure/search/agentic-knowledge-source-how-to-fabric-ontology ·
https://learn.microsoft.com/en-us/azure/search/agentic-retrieval-how-to-create-knowledge-base ·
https://learn.microsoft.com/en-us/azure/search/agentic-retrieval-how-to-retrieve

In [20]:
def aoai(deployment, model):
    return {"kind": "azureOpenAI", "azureOpenAIParameters": {"resourceUri": FOUNDRY_OPENAI_URL,
                                                             "deploymentId": deployment, "modelName": model}}

ks_docs = {
    "name": KS_DOCS, "kind": "searchIndex",
    "description": "Venue manuals, inspection reports, transport plans and policies, chunked, with document status.",
    "searchIndexParameters": {"searchIndexName": INDEX, "semanticConfigurationName": "games-semantic",
                              "sourceDataFields": [{"name": f} for f in ("document_id", "title", "document_type", "venue_id",
                                                                         "version", "document_status", "page_from")]},
}
ks_onelake = {
    "name": KS_ONELAKE, "kind": "indexedOneLake",
    "description": "The same PDF folder, indexed by a generated pipeline.",
    "indexedOneLakeParameters": {"fabricWorkspaceId": FABRIC_WORKSPACE_ID, "lakehouseId": FABRIC_LAKEHOUSE_ID,
                                 "targetPath": FABRIC_DOCS_FOLDER,
                                 "ingestionParameters": {"embeddingModel": aoai(FOUNDRY_EMBEDDING_NAME, EMBED_MODEL),
                                                         "chatCompletionModel": aoai("gpt-5.4-mini","gpt-5.4-mini"),
                                                         "contentExtractionMode": "minimal"}},
}
ks_ontology = {
    "name": KS_ONTOLOGY, "kind": "fabricOntology",
    "description": "Games operations ontology: venues, sessions, athletes, assets, findings, rules and live readings.",
    "fabricOntologyParameters": {"workspaceId": FABRIC_WORKSPACE_ID, "ontologyId": FABRIC_ONTOLOGY_ID},
}
if LIVE:
    index_client.create_or_update_knowledge_source(ks_docs)
    index_client.create_or_update_knowledge_source(ks_ontology)   # preview kind: needs the 2026-08-01-preview API version
    # CHARGEABLE: the indexed OneLake source runs its own extraction and embeddings over the 51 PDFs
    index_client.create_or_update_knowledge_source(ks_onelake)
    print("Knowledge sources created:", KS_DOCS, KS_ONTOLOGY, KS_ONELAKE)

Knowledge sources created: games-docs-ks games-ontology-ks games-onelake-ks


The indexed OneLake source builds its pipeline from a fixed template and names the objects after itself
(`games-onelake-ks-datasource`, `-skillset`, `-index`, `-indexer`). The next cell waits for the first
synchronisation and lists the generated index fields, to compare them with the step 3 index.

In [21]:
if LIVE:
    t0 = time.time()
    while time.time() - t0 < 1800:
        st = index_client.get_knowledge_source_status(KS_ONELAKE).as_dict()   # verify https://learn.microsoft.com/en-us/azure/search/agentic-knowledge-source-how-to-onelake
        last = st.get("lastSynchronizationState") or st.get("last_synchronization_state")
        print(f"  {int(time.time() - t0)} s · {st.get('synchronizationStatus') or st.get('synchronization_status')}", end="\r")
        if last:
            print("\nLast synchronisation:", json.dumps(last, default=str)[:400])
            break
        time.sleep(20)
    generated = index_client.get_index(f"{KS_ONELAKE}-index")
    mine = {f.name for f in index_definition.fields}
    theirs = {f.name for f in generated.fields}
    print("Generated index fields:", ", ".join(sorted(theirs)))
    print("Only in the step 3 index:", ", ".join(sorted(mine - theirs)))

  0 s · active
Last synchronisation: {"startTime": "2026-11-09T21:10:58.501Z", "endTime": "2026-11-09T21:11:10.023Z", "itemsUpdatesProcessed": 51, "itemsUpdatesFailed": 0, "itemsSkipped": 0, "errors": []}
Generated index fields: doc_url, image_snippet_parent_id, snippet, snippet_parent_id, snippet_vector, uid
Only in the step 3 index: chunk, chunk_id, document_id, document_status, document_type, file_name, page_from, page_to, parent_id, sensitivity, text_vector, title, venue_id, version


One knowledge base over the three sources. The retrieval instructions tell the planner which source answers
which kind of question; the answer instructions ask for citations and for superseded documents to be ignored.

In [22]:
kb_definition = {
    "name": KB,
    "description": "Games operations knowledge: ontology plus documents.",
    "knowledgeSources": [{"name": KS_DOCS}, {"name": KS_ONELAKE}, {"name": KS_ONTOLOGY}],
    "models": [aoai("gpt-5.4-mini", "gpt-5.4-mini")],
    "outputMode": "answerSynthesis",
    "retrievalReasoningEffort": {"kind": "low"},
    "retrievalInstructions": ("Use games-ontology-ks for counts, schedules, athletes, assets, findings, business rules and "
                              "live readings. Use the document sources for procedures, operating limits and inspector wording."),
    "answerInstructions": ("Answer in at most five sentences and cite every source. Only documents with status Current are "
                           "authoritative; if a source is Superseded, say so and don't use its values."),
}
if LIVE:
    index_client.create_or_update_knowledge_base(kb_definition)
    print("Knowledge base", KB, "ready")

Knowledge base games-kb ready


**Define retrieval for the comparison cells.** This cell sets up the SDK client and function; it does not run
a query or display results. The following cells call `retrieve` with a question and selected sources, then display
the answer, cited documents, reported tokens and elapsed time. The ontology source uses your own sign-in, scoped
to Azure AI Search and passed through the SDK's `headers` argument as `x-ms-query-source-authorization`.
Retrieval errors stop execution with a traceback rather than appearing as answers in the comparison table.

In [23]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient

SOURCE_KIND = {KS_DOCS: "searchIndex", KS_ONELAKE: "indexedOneLake", KS_ONTOLOGY: "fabricOntology"}
CURRENT_ONLY = {KS_DOCS: "document_status eq 'Current'"}

def pick(d, name, default=None):
    snake = re.sub(r"(?<!^)(?=[A-Z])", "_", name).lower()
    return d.get(name, d.get(snake, default)) if isinstance(d, dict) else default

def ref_label(ref):
    data = pick(ref, "sourceData") or {}
    if pick(ref, "type") == "fabricOntology":
        return "ontology"
    return (pick(data, "document_id") or pick(data, "title") or pick(ref, "docKey") or
            (pick(ref, "docUrl") or "").rsplit("/", 1)[-1] or "?")

def retrieve(question, sources, effort="low", output="answerSynthesis", filters=None):
    params = [{"knowledgeSourceName": s, "kind": SOURCE_KIND[s], "includeReferences": True,
               "includeReferenceSourceData": True, **({"filterAddOn": filters[s]} if filters and s in filters else {})}
              for s in sources]
    body = {"messages": [{"role": "user", "content": [{"type": "text", "text": question}]}],
            "knowledgeSourceParams": params, "retrievalReasoningEffort": {"kind": effort},
            "outputMode": output, "includeActivity": True}
    user_token = credential.get_token("https://search.azure.com/.default").token
    t0 = time.perf_counter()
    result = kb_client.retrieve(
        retrieval_request=body,
        headers={"x-ms-query-source-authorization": user_token},
    ).as_dict()
    seconds = round(time.perf_counter() - t0, 1)
    answer = " ".join(pick(c, "text", "") for m in pick(result, "response", []) for c in pick(m, "content", []))
    tokens = sum((pick(a, "inputTokens") or 0) + (pick(a, "outputTokens") or 0) + (pick(a, "reasoningTokens") or 0)
                 for a in pick(result, "activity", []))
    cited = sorted({ref_label(r) for r in pick(result, "references", [])})
    return {"answer": answer[:400], "cited": ", ".join(cited)[:160], "tokens": tokens, "seconds": seconds}

if LIVE:
    kb_client = KnowledgeBaseRetrievalClient(endpoint=AZURE_SEARCH_ENDPOINT, knowledge_base_name=KB,
                                             credential=credential, api_version=SEARCH_API_VERSION)

**Documents: custom index, custom index filtered to Current documents, generated OneLake index.** Q09 and Q13 ask
for wind limits that differ between the Current and the Superseded manual; Q10 asks for a procedure; Q17 asks for
an inspector's wording.

In [19]:
pd.set_option("display.max_colwidth", 400)
DOC_RUNS = [("custom index", [KS_DOCS], None), ("custom index, Current only", [KS_DOCS], CURRENT_ONLY),
            ("generated OneLake index", [KS_ONELAKE], None)]
if LIVE:
    # CHARGEABLE: 12 knowledge base requests (query planning and answer synthesis)
    rows = []
    for qid in ("Q09", "Q13", "Q10", "Q17"):
        for label, sources, flt in DOC_RUNS:
            rows.append({"question": qid, "source": label, **retrieve(questions.loc[qid].question, sources, filters=flt)})
    doc_results = pd.DataFrame(rows)
    display(doc_results)
    display(questions.loc[["Q09", "Q13", "Q10", "Q17"], ["question", "expected_answer"]])

,question,source,answer,cited,tokens,seconds
0,Q09,custom index,"Summit Alpine Centre’s wind gust limit is 15.0 m/s, according to the current operations manual for the venue. [ref_id:1]","DOC-IR-0018, DOC-IR-0028, DOC-IR-0032, DOC-MAN-V07-10, DOC-MAN-V07-21, DOC-MAN-V08-10, DOC-MAN-V08-21, DOC-MAN-V09-21, DOC-MAN-V10-21, ontology",41829,21.5
1,Q09,"custom index, Current only","The current Summit Alpine Centre operations manual states that competition must not start, and must be interrupted, when the wind gust exceeds 15.0 m/s for three consecutive minutes. [ref_id:3] The current inspection report for Summit Alpine Centre is also marked Current, but it does not state a wind gust limit. [ref_id:15]","?, DOC-IR-0018, DOC-IR-0028, DOC-IR-0032, DOC-MAN-V07-21, DOC-MAN-V08-21, DOC-MAN-V09-21, DOC-MAN-V10-21, ontology",67239,21.5
2,Q09,generated OneLake index,"The current Summit Alpine Centre operations manual states that competition must not start, and must be interrupted, when wind gusts exceed 15.0 m/s for three consecutive minutes. [ref_id:3]",2daffad7705d_aHR0cHM6Ly9vbmVsYWtlLmJsb2IuZmFicmljLm1pY3Jvc29mdC5jb20vNmZjOTY2NTctODRlNS00ZGNhLWEyMTUtN2ZhMTVkMGJkYzk3LzJmZTU1NTMzLTJhMDgtNDY4My05Y2ZkLTE0MDhhNzQ,71377,17.5
3,Q13,custom index,Ridge Ski Jumping Hill’s wind gust limit is 4.5 m/s. [ref_id:3],"DOC-IR-0007, DOC-IR-0017, DOC-IR-0022, DOC-IR-0025, DOC-MAN-V05-21, DOC-MAN-V06-21, DOC-MAN-V07-10, DOC-MAN-V07-21, DOC-MAN-V08-10, DOC-MAN-V08-21, DOC-MAN-V09-",35187,34.4
4,Q13,"custom index, Current only",The wind gust limit for Ridge Ski Jumping Hill is 4.5 m/s. [ref_id:1],"DOC-IR-0007, DOC-IR-0017, DOC-IR-0022, DOC-IR-0025, DOC-MAN-V05-21, DOC-MAN-V06-21, DOC-MAN-V07-21, DOC-MAN-V08-21, DOC-MAN-V09-21, DOC-MAN-V10-21, ontology",31210,24.2
5,Q13,generated OneLake index,"For Ridge Ski Jumping Hill, the wind gust limit is 4.5 m/s. [ref_id:3]",2daffad7705d_aHR0cHM6Ly9vbmVsYWtlLmJsb2IuZmFicmljLm1pY3Jvc29mdC5jb20vNmZjOTY2NTctODRlNS00ZGNhLWEyMTUtN2ZhMTVkMGJkYzk3LzJmZTU1NTMzLTJhMDgtNDY4My05Y2ZkLTE0MDhhNzQ,35190,15.8
6,Q10,custom index,"When a gate plaza exceeds the 4.0 persons per square metre limit for five minutes or more, the gate supervisor must open the overflow lane, the transport hub must hold arriving shuttles for 10 minutes, and a public address announcement must redirect spectators to the nearest gate with spare capacity. [ref_id:2] The Games-wide standard also says breaches must be reported to the Main Operations ...","?, DOC-MAN-V01-21, DOC-MAN-V02-21, DOC-MAN-V03-21, DOC-MAN-V04-21, DOC-MAN-V05-21, DOC-MAN-V07-10, DOC-MAN-V08-10, DOC-MAN-V08-21, DOC-MAN-V09-21, DOC-MAN-V10-2",68292,16.0
7,Q10,"custom index, Current only","When a gate plaza exceeds the crowd density limit for five minutes or more, the gate supervisor opens the overflow lane, the transport hub holds arriving shuttles for 10 minutes, and a public address announcement redirects spectators to the nearest gate with spare capacity. [ref_id:1] The Games-wide standard also says breaches must be reported to the Main Operations Centre within 10 minutes wi...","DOC-IR-0001, DOC-IR-0004, DOC-IR-0005, DOC-IR-0020, DOC-IR-0027, DOC-IR-0031, DOC-IR-0032, DOC-MAN-V01-21, DOC-MAN-V02-21, DOC-MAN-V03-21, DOC-MAN-V04-21, DOC-M",38825,17.0
8,Q10,generated OneLake index,"When a gate plaza exceeds the crowd density limit for five minutes or more, the gate supervisor must open the overflow lane, the transport hub must hold arriving shuttles for 10 minutes, and a public address announcement must redirect spectators to the nearest gate with spare capacity. [ref_id:2][ref_id:20] The Games-wide standard also says the breach must be reported to the Main Operations Ce...",82c5c3d2a69a_aHR0cHM6Ly9vbmVsYWtlLmJsb2IuZmFicmljLm1pY3Jvc29mdC5jb20vNmZjOTY2NTctODRlNS00ZGNhLWEyMTUtN2ZhMTVkMGJkYzk3LzJmZTU1NTMzLTJhMDgtNDY4My05Y2ZkLTE0MDhhNzQ,68298,17.7
9,Q17,custom index,"The 23 November 2026 inspection of Summit Sliding Centre found 1 current issue, classified as High in Technical systems.[ref_id:2] It 

,question,expected_answer
id,,
Q09,What is the wind gust limit for Summit Alpine Centre?,"15.0 m/s for three consecutive minutes, from the current manual DOC-MAN-V08-21 (version 2.1); the superseded version 1.0 must not be used"
Q13,What is the wind gust limit for Ridge Ski Jumping Hill?,"4.5 m/s for three consecutive minutes, from the current manual DOC-MAN-V07-21 (version 2.1); the superseded version 1.0 must not be used"
Q10,What must happen when a gate plaza exceeds the crowd density limit?,Open the overflow lane at the gate; the transport hub holds arriving shuttles for 10 minutes; a public address announcement redirects spectators to a gate with spare capacity; report to the Main Operations Centre within 10 minutes
Q17,What did the 23 November 2026 inspection of Summit Sliding Centre find?,"F-0034-1 (High, Technical systems): Brine return temperature on the track refrigeration plant has risen steadily since the previous day; compressor 2 shows vibration above its baseline. Recommendation: Replace the compressor 2 bearing before the next sliding session and run the plant on the redundant compressor until the repair is complete."


**Ontology source.** Q05 joins sessions and venue conditions, Q12 needs the `ServiceOverdue` rule, and Q11 needs
the findings extracted by AI functions and the `SessionAtRisk` rule.

In [20]:
if LIVE:
    # CHARGEABLE: 3 knowledge base requests
    rows = [{"question": qid, **retrieve(questions.loc[qid].question, [KS_ONTOLOGY])} for qid in ("Q05", "Q12", "Q11")]
    display(pd.DataFrame(rows))
    display(questions.loc[["Q05", "Q12", "Q11"], ["question", "expected_answer"]])

,question,answer,cited,tokens,seconds
0,Q05,"The postponed competition session was S-V07-D03-3 on 2026-11-18. [ref_id:0] It was postponed because wind gusts were above the venue operating limit. [ref_id:0] It was rescheduled for 2026-11-19 at 19:00 UTC, ending at 20:45 UTC, as S-V07-D04-R. [ref_id:0]",ontology,3411,25.8
1,Q12,"I found 1 Critical asset overdue for service by more than 30 days: V08-AS-020 (Safety netting section 1), last serviced on 2026-10-07 00:00:00 UTC, which is 34 days ago. [ref_id:0]",ontology,3397,25.6
2,Q11,"One scheduled session is at risk: S-V10-D10-1 at Summit Sliding Centre (V10), scheduled for 2026-11-25 from 09:30 to 11:30 UTC, because it is exposed by an open High-severity finding, F-0034-1, on the critical asset Track refrigeration plant (V10-AS-018). [ref_id:0] The inspector recommended replacing compressor 2 bearing before the next sliding session and running the plant on the redundant c...",ontology,3540,54.2


,question,expected_answer
id,,
Q05,"Which competition session was postponed, why, and when was it rescheduled?","S-V07-D03-3 (Men's Normal Hill Individual, Ridge Ski Jumping Hill, 18 November 2026) was postponed because wind gusts exceeded the venue operating limit; rescheduled as S-V07-D04-R on 19 November 2026 at 19:00 UTC"
Q12,Which Critical assets are overdue for service (more than 30 days since the last service)?,"V08-AS-020 (Safety netting section, Summit Alpine Centre, last service 2026-10-07)"
Q11,"Which scheduled sessions are at risk from open High-severity findings on Critical assets, and what did the inspector recommend?","S-V10-D10-1 (Women's Monobob, 25 November 2026); S-V10-D12-1 (Two-man Bobsleigh, 27 November 2026). Recommendation: Replace the compressor 2 bearing before the next sliding session and run the plant on the redundant compressor until the repair is complete."


**Reasoning effort.** Q11 needs both the ontology (which sessions) and the inspection report (what the inspector
recommended). The same question runs with the custom index (Current only) plus the ontology at *low* and
*medium* effort; *minimal* runs on the documents alone, because it needs extractive output and the ontology source
doesn't support it. Higher effort plans more sub-queries: compare citations, tokens and time.

In [22]:
if LIVE:
    # CHARGEABLE: 3 knowledge base requests
    q11 = questions.loc["Q11"].question
    rows = [{"effort": e, **retrieve(q11, [KS_DOCS, KS_ONTOLOGY], effort=e, filters=CURRENT_ONLY)} for e in ("low", "medium")]
    rows.append({"effort": "low (documents only, extractive)",
                 **retrieve(q11, [KS_DOCS], effort="low", output="extractiveData", filters=CURRENT_ONLY)})
    display(pd.DataFrame(rows))
    print("Expected:", questions.loc["Q11"].expected_answer)

,effort,answer,cited,tokens,seconds
0,low,Two scheduled sessions are at risk: **S-V10-D10-1** on **2026-11-25 09:30–11:30 UTC** and **S-V10-D12-1** on **2026-11-27 09:30–11:30 UTC**. [ref_id:0] Both are at **Summit Sliding Centre (V10)**. [ref_id:0] The inspector recommended: **replace compressor 2 bearing before the next sliding session and run the plant on the redundant compressor until repair complete**. [ref_id:0],ontology,3543,130.2
1,medium,"I found one open High-severity finding on a Critical asset: at Summit Sliding Centre, finding F-0034-1 concerns asset V10-AS-018, and the inspector recommended replacing compressor 2 bearing before the next sliding session and running the plant on the redundant compressor until repair is complete. [ref_id:23] The retrieved content also states that a High-severity finding on a Critical asset mu...","?, DOC-IR-0001, DOC-IR-0002, DOC-IR-0004, DOC-IR-0005, DOC-IR-0007, DOC-IR-0008, DOC-IR-0013, DOC-IR-0014, DOC-IR-0015, DOC-IR-0016, DOC-IR-0018, DOC-IR-0021, D",118713,83.3
2,"low (documents only, extractive)","[{""ref_id"":0,""content"":""Two scheduled sessions are at risk at Summit Sliding Centre (venue V10) because of one open High-severity finding on the critical asset Track refrigeration plant (asset V10-AS-018). The affected sessions are S-V10-D10-1 on 2026-11-25 09:30–11:30 UTC and S-V10-D12-1 on 2026-11-27 09:30–11:30 UTC. The inspector recommended: \""Replace compressor 2 bearing before next slidi...","?, DOC-IR-0005, DOC-IR-0009, DOC-IR-0015, DOC-IR-0018, DOC-IR-0019, DOC-IR-0026, DOC-IR-0027, DOC-IR-0028, DOC-IR-0031, DOC-IR-0032, DOC-IR-0033, DOC-MAN-V01-21",95533,43.6


Expected: S-V10-D10-1 (Women's Monobob, 25 November 2026); S-V10-D12-1 (Two-man Bobsleigh, 27 November 2026). Recommendation: Replace the compressor 2 bearing before the next sliding session and run the plant on the redundant compressor until the repair is complete.


**What to look for**
- The filter on `document_status` is what keeps the superseded wind limits out of Q09 and Q13. The generated
  index has no register fields, so it can't apply that filter.
- The ontology source answers with data and returns the raw rows behind the answer; the document sources answer
  with cited chunks.
- From step 5 onward, the agents use the custom index (Current only) and the ontology. Using both document sources
  at once would return duplicate chunks of the same files.

## 5 · Three Foundry agent patterns (preview)  *(deck sections 5, 6)*

The same questions run through three agents built on the knowledge layer from steps 1 to 4. They differ in who runs
the agent loop and in how each one reaches the ontology and the documents:

| Agent | Pattern | Runtime | Route to the ontology and the documents |
|---|---|---|---|
| `games-ops-prompt` | Prompt agent | Foundry Agent Service | Toolbox `games-ops-tools`: the Fabric IQ ontology tool and the `games-agent-kb` knowledge base |
| `games-ops-harness` | Prompt agent with the Microsoft Copilot harness | Foundry-operated harness, one sandbox per conversation | The same toolbox; the shell and web harness tools are turned off |
| `games-ops-hosted` | Hosted agent (Microsoft Agent Framework) | This notebook's Python process; deployable as a hosted agent | Its own function tools: the ontology MCP endpoint, and the knowledge base filtered to Current documents |

The prompt agents call the toolbox with the caller's identity (a `UserEntraToken` connection), and Fabric IQ runs
the ontology query as that user. The hosted agent runs here as you. Deployed, it would run as its own agent identity,
which the ontology endpoint doesn't accept; `samples/games-operations/hosted-agent/README.md` lists the production
options.

The agents get their own knowledge base, `games-agent-kb`, over the step 3 index only: they reach the ontology
through Fabric IQ, so the ontology is queried in one place and the traces show one call per source. Extractive
output with *minimal* reasoning returns passages and leaves the answer to the agent.

Sources: https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/fabric-iq ·
https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/toolbox ·
https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/foundry-iq-connect ·
https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/use-tools-skills-copilot-harness ·
https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/use-toolbox-hosted-agent

### Agent settings

Names of everything steps 5 and 6 create, the project's Azure resource ID (for connections and role assignments)
and the instructions shared by the two prompt agents. `preview_project` is the client for preview surfaces: the
Copilot harness and agent-scoped Responses clients.

In [24]:
from azure.ai.projects.models import PromptAgentDefinition, MCPTool, FabricIQPreviewToolboxTool, MCPToolboxTool

AZURE_SUBSCRIPTION_ID = env("AZURE_SUBSCRIPTION_ID")
AZURE_RESOURCE_GROUP = env("AZURE_RESOURCE_GROUP")           # resource group of the Foundry account, as in notebook 1
FOUNDRY_PROJECT_NAME = env("FOUNDRY_PROJECT_NAME") or urlparse(FOUNDRY_PROJECT_ENDPOINT).path.rstrip("/").split("/")[-1]
CONNECTIONS_API_VERSION = env("FOUNDRY_CONNECTIONS_API_VERSION", "2025-10-01-preview")   # ARM project connections (RemoteTool)
PROJECT_RESOURCE_ID = (f"/subscriptions/{AZURE_SUBSCRIPTION_ID}/resourceGroups/{AZURE_RESOURCE_GROUP}"
                       f"/providers/Microsoft.CognitiveServices/accounts/{ACCOUNT}/projects/{FOUNDRY_PROJECT_NAME}")
AGENTS_LIVE = LIVE and all(value and "<" not in value for value in (AZURE_SUBSCRIPTION_ID, AZURE_RESOURCE_GROUP))

AGENT_KB = "games-agent-kb"
AGENT_KB_MCP_URL = f"{AZURE_SEARCH_ENDPOINT.rstrip('/')}/knowledgebases/{AGENT_KB}/mcp?api-version={SEARCH_API_VERSION}"
CONN_ONTOLOGY, CONN_DOCS, CONN_TOOLBOX = "games-ontology-conn", "games-agent-kb-conn", "games-ops-tools-conn"
TOOLBOX, ONTOLOGY_LABEL, DOCS_LABEL = "games-ops-tools", "games-ontology", "games-docs"
PROMPT_AGENT, HARNESS_AGENT, HOSTED_AGENT = "games-ops-prompt", "games-ops-harness", "games-ops-hosted"
COPILOT_HEADERS = {"Foundry-Features": "MicrosoftCopilot=V1Preview"}

AGENT_INSTRUCTIONS = """You are the operations assistant of the Valmont Winter Games operations centre.
Use the games-ontology tool for venues, sessions, athletes, assets, findings, business rules and live readings.
Use the games-docs knowledge base for procedures, operating limits and inspector wording.
For questions requiring both operational facts and document evidence:

1. Do not forward the original mixed question to the ontology.
   Rewrite it into an ontology-only question about entities,
   relationships and status. Request the relevant session, asset,
   finding and source report identifiers.
   Explicitly exclude document retrieval and inspector recommendations.

2. After receiving the ontology result, make a separate games-docs
   call using the returned finding and report identifiers.
   Retrieve the relevant recommendation or procedure.
   Use only passages whose document_status is Current.
   Never invent identifiers or substitute an unrelated report.

3. Combine the operational facts and document evidence in the final
   answer, citing the source report. If either lookup fails, explain
   which part could not be verified.
Cite every document by its document ID (for example DOC-MAN-V07-21). Answer in at most six sentences.
If the tools don't return the answer, say you don't know."""

if AGENTS_LIVE:
    preview_project = AIProjectClient(endpoint=FOUNDRY_PROJECT_ENDPOINT, credential=credential, allow_preview=True)
    print("Project resource ID:", PROJECT_RESOURCE_ID)
else:
    print("AZURE_SUBSCRIPTION_ID or AZURE_RESOURCE_GROUP not set (or step 0 settings missing) — steps 5 and 6 skipped")

Project resource ID: /subscriptions/dcbc681e-a69d-4f95-bc8e-da6054697474/resourceGroups/rg-rag-telemetry/providers/Microsoft.CognitiveServices/accounts/cog-tb7tpjtuee4ji/projects/cog-tb7tpjtuee4ji-project


### Agent knowledge base and project connections

The agent knowledge base lists only the step 3 documents source. Three project connections carry the credentials, so
no token or key appears in an agent definition:

| Connection | Target | Authentication | Used by |
|---|---|---|---|
| `games-ontology-conn` | Ontology MCP endpoint (step 1) | User Entra token, audience Power BI | Fabric IQ tool in the toolbox |
| `games-agent-kb-conn` | `games-agent-kb` MCP endpoint | Project managed identity, audience Azure AI Search | Knowledge base tool in the toolbox |
| `games-ops-tools-conn` | Toolbox MCP endpoint (next cell) | User Entra token, audience `https://ai.azure.com` | Both prompt agents |

The connections are created with an Azure Resource Manager (ARM) REST call, the pattern shown in the official
documentation for these connection types; the API version comes from `FOUNDRY_CONNECTIONS_API_VERSION`. As per
official documentation, the ontology accepts the delegated Power BI permissions `Item.Execute.All` and
`Item.Read.All`, which is why its audience is Power BI rather than the Fabric API.

Sources: https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/foundry-iq-connect ·
https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/fabric-iq

In [25]:
agent_kb = {
    "name": AGENT_KB,
    "description": "Venue manuals, inspection reports, transport plans and policies, for the games operations agents.",
    "knowledgeSources": [{"name": KS_DOCS}],
    "outputMode": "extractiveData",
    "retrievalReasoningEffort": {"kind": "minimal"},
}
arm = httpx.Client(base_url=f"https://management.azure.com{PROJECT_RESOURCE_ID}/connections",
                   params={"api-version": CONNECTIONS_API_VERSION}, timeout=60)

def put_connection(name, **properties):
    token = credential.get_token("https://management.azure.com/.default").token
    response = arm.put(f"/{name}", headers={"Authorization": f"Bearer {token}"},
                       json={"properties": {"category": "RemoteTool", "isSharedToAll": True, **properties}})
    response.raise_for_status()
    return response.json()["id"]

if AGENTS_LIVE:
    index_client.create_or_update_knowledge_base(agent_kb)
    ontology_connection_id = put_connection(CONN_ONTOLOGY, authType="UserEntraToken", target=ONTOLOGY_MCP_URL,
                                            audience="https://analysis.windows.net/powerbi/api")   # verify https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/fabric-iq
    put_connection(CONN_DOCS, authType="ProjectManagedIdentity", target=AGENT_KB_MCP_URL,
                   audience="https://search.azure.com/", metadata={"ApiType": "Azure"})
    print("Knowledge base", AGENT_KB, "and connections", CONN_ONTOLOGY, CONN_DOCS, "ready")

Knowledge base games-agent-kb and connections games-ontology-conn games-agent-kb-conn ready


### One toolbox for the prompt agents

A toolbox bundles tool configurations behind one MCP endpoint, so both prompt agents use the same versioned tool set.
This cell creates a new toolbox version, points the toolbox connection at it and lists the tools the endpoint
exposes. Listing tools doesn't call the ontology or the knowledge base.

Source: https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/toolbox

In [26]:
if AGENTS_LIVE:
    toolbox = project.toolboxes.create_version(
        name=TOOLBOX,
        description="Games operations: ontology facts and operating documents.",
        tools=[
            FabricIQPreviewToolboxTool(project_connection_id=ontology_connection_id, server_label=ONTOLOGY_LABEL,
                                       server_url=ONTOLOGY_MCP_URL, require_approval="never"),   # verify https://learn.microsoft.com/en-us/python/api/azure-ai-projects/azure.ai.projects.models.fabriciqpreviewtool
            MCPToolboxTool(server_label=DOCS_LABEL, server_url=AGENT_KB_MCP_URL, require_approval="never",
                           project_connection_id=CONN_DOCS),
        ],
    )
    TOOLBOX_MCP_URL = f"{FOUNDRY_PROJECT_ENDPOINT}/toolboxes/{toolbox.name}/versions/{toolbox.version}/mcp?api-version=v1"
    put_connection(CONN_TOOLBOX, authType="UserEntraToken", target=TOOLBOX_MCP_URL, audience="https://ai.azure.com")

    headers = {"Authorization": f"Bearer {credential.get_token('https://ai.azure.com/.default').token}"}
    async with streamablehttp_client(TOOLBOX_MCP_URL, headers=headers) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            toolbox_tools = (await session.list_tools()).tools
    print(f"Toolbox {toolbox.name}, version {toolbox.version}")
    display(pd.DataFrame([{"tool": t.name, "description": (t.description or "")[:160]} for t in toolbox_tools]))

Toolbox games-ops-tools, version 3


,tool,description
0,games-docs___knowledge_base_retrieve,Use knowledge_base_retrieve to search for auth...
1,games-ontology___ask_ontology,Answers day-to-day business questions using an...
2,games-ontology___list_ontology_entities,"Lists the entities defined in the ontology, in..."
3,games-ontology___list_ontology_rules,"Lists the business rule definitions, including..."


### 5a · Prompt agent

A prompt agent is a definition: model, instructions and tools. Foundry runs the loop, and the model decides which
toolbox tool to call. The toolbox is attached as one MCP tool through its connection.

Source: https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/fabric-iq

In [27]:
if AGENTS_LIVE:
    toolbox_tool = MCPTool(server_label=TOOLBOX, server_url=TOOLBOX_MCP_URL, require_approval="never",
                           project_connection_id=CONN_TOOLBOX)
    prompt_agent = project.agents.create_version(
        agent_name=PROMPT_AGENT,
        definition=PromptAgentDefinition(model=FOUNDRY_MODEL_NAME, instructions=AGENT_INSTRUCTIONS, tools=[toolbox_tool]),
    )
    print(f"Agent {prompt_agent.name}, version {prompt_agent.version}")

Agent games-ops-prompt, version 3


### 5b · Prompt agent with the Copilot harness (preview)

The same definition with the Microsoft Copilot harness as its runtime: Foundry operates a multi-step runtime with a
workspace and sandbox per conversation, and adds harness tools (file system, shell, web, subagents). The harness is
chosen when the agent is created and can't be added to an existing agent. Shell and web are turned off here, so the
agent works only with the toolbox and its workspace files. Each request needs a conversation.

The `MicrosoftCopilot...` Python model classes require `azure-ai-projects` 2.9 or later. To also run with the
installed 2.8 SDK without upgrading the shared environment, this cell uses the SDK's supported JSON `body`
argument and an explicit Microsoft Copilot preview header. It preserves the `microsoft_copilot_preview` harness
and disabled shell/web tools; it does not switch to the older GitHub harness. The same header is passed when
calling the harness agent below. No package installation is needed for this compatibility fix.

Source: https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/use-tools-skills-copilot-harness

In [28]:
if AGENTS_LIVE:
    harness_agent = preview_project.agents.create_version(
        agent_name=HARNESS_AGENT,
        body={"definition": {
            "kind": "prompt", "model": FOUNDRY_MODEL_NAME, "instructions": AGENT_INSTRUCTIONS,
            "harness": {"type": "microsoft_copilot_preview"},
            "tools": [toolbox_tool.as_dict(), {
                "type": "microsoft_copilot_toolset_preview",
                "configs": [{"name": "shell", "enabled": False}, {"name": "web", "enabled": False}],
            }],
        }},
        headers=COPILOT_HEADERS,
    )
    print(f"Agent {harness_agent.name}, version {harness_agent.version}")

Agent games-ops-harness, version 3


### Agent identities

As per official documentation, a prompt agent that calls a toolbox needs the **Foundry User** role on the project for
its own identity (`instance_identity`), separate from the user's role. This cell assigns it to both agents with the
role's ID. Allow up to five minutes for the assignment to take effect before the first question.

Source: https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/use-tools-skills-copilot-harness

In [29]:
from uuid import uuid4
from azure.core.exceptions import HttpResponseError
from azure.mgmt.authorization import AuthorizationManagementClient
from azure.mgmt.authorization.models import RoleAssignmentCreateParameters

FOUNDRY_USER_ROLE = "53ca6127-db72-4b80-b1b0-d745d6d5456d"
if AGENTS_LIVE:
    authorization = AuthorizationManagementClient(credential, AZURE_SUBSCRIPTION_ID)
    role_definition_id = f"/subscriptions/{AZURE_SUBSCRIPTION_ID}/providers/Microsoft.Authorization/roleDefinitions/{FOUNDRY_USER_ROLE}"
    for name in (PROMPT_AGENT, HARNESS_AGENT):
        principal_id = (project.agents.get(agent_name=name).as_dict().get("instance_identity") or {}).get("principal_id")   # verify https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/use-tools-skills-copilot-harness
        if not principal_id:
            print(f"{name}: no instance identity returned; assign Foundry User to the agent identity in the Azure portal")
            continue
        try:
            authorization.role_assignments.create(PROJECT_RESOURCE_ID, str(uuid4()), RoleAssignmentCreateParameters(
                role_definition_id=role_definition_id, principal_id=principal_id, principal_type="ServicePrincipal"))
            print(f"{name}: Foundry User assigned to {principal_id}")
        except HttpResponseError as e:
            if not (e.error and e.error.code == "RoleAssignmentExists"):
                raise
            print(f"{name}: Foundry User already assigned")

games-ops-prompt: Foundry User already assigned
games-ops-harness: Foundry User already assigned


### 5c · Hosted agent (Microsoft Agent Framework)

The hosted agent is code: `samples/games-operations/hosted-agent/main.py`. It has two function tools whose behaviour
the development team controls: `ask_ontology` calls the ontology MCP endpoint as the caller, and
`search_current_documents` retrieves passages from `games-agent-kb` with the filter `document_status eq 'Current'`.
A filter per request isn't possible through the knowledge base MCP tool that the prompt agents use. The model still
decides when to call each tool.

The notebook imports the module and runs the agent in this process, as you; the same file deploys with
`azd up` from that folder (see its README for the identity options).

Source: https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/use-toolbox-hosted-agent

In [30]:
import importlib.util

# main.py reads its settings from the environment loaded in step 0; its knowledge base and source names default to
# games-agent-kb and games-docs-ks, the names used in this notebook.
if AGENTS_LIVE:
    spec = importlib.util.spec_from_file_location("games_ops_hosted", SAMPLE / "hosted-agent" / "main.py")
    hosted = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(hosted)
    print(f"Hosted agent {hosted.agent.name} · tools: ask_ontology, search_current_documents")

Hosted agent games-ops-hosted · tools: ask_ontology, search_current_documents


### Tracing on

Prompt agents are traced on the server side automatically. Client-side tracing (preview) adds this notebook's calls
and the in-process hosted agent: opt in, export to the project's Application Insights, instrument the project client
and Agent Framework. Run this cell once per kernel, before the questions. Prompts and outputs stay out of the traces
(`OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT=false`).

Sources: https://learn.microsoft.com/en-us/azure/foundry/observability/how-to/trace-agent-client-side ·
https://learn.microsoft.com/en-us/agent-framework/agents/observability

In [31]:
from azure.ai.projects.telemetry import AIProjectInstrumentor
from azure.monitor.opentelemetry import configure_azure_monitor

if AGENTS_LIVE:
    os.environ["AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING"] = "true"
    os.environ.setdefault("OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT", "false")
    configure_azure_monitor(connection_string=project.telemetry.get_application_insights_connection_string())
    AIProjectInstrumentor().instrument()
    from agent_framework.observability import enable_instrumentation   # verify https://learn.microsoft.com/en-us/agent-framework/agents/observability
    enable_instrumentation()
    print("Client-side tracing on; spans reach Application Insights within 2 to 5 minutes")

Client-side tracing on; spans reach Application Insights within 2 to 5 minutes


### Same questions, three agents

Q05 needs the ontology, Q09 the documents (with a superseded manual as a trap), and Q11 both. Each row reports the
tool calls, the documents cited in the answer, whether a superseded document was cited, tokens and time. Prompt
agents are called through their agent endpoint with a new conversation per question.

Each run has a **180-second client deadline**, with automatic SDK request retries disabled. Calls remain sequential,
but a timeout or service error is recorded separately from the answer and the next agent still runs. This does not
change the Fabric/harness service timeout or guarantee that remote work stops when the client disconnects.
The in-process hosted tools use asynchronous I/O so cancellation can reach them; MCP requests also have a 90-second deadline.

Every start and result is printed and appended to a unique `output/eval/nb9-agent-comparison-*.jsonl` file under the
sample. The path is printed before calls begin. After a kernel restart, recover the latest row for each pair with
`pd.read_json(path, lines=True).drop_duplicates(['question', 'agent'], keep='last')`; a remaining `running` row means
no terminal result was saved. `completed` means a response returned, not that its answer is correct; inspect tool errors
and traces too. Edit `SHOWCASE` and `SHOWCASE_AGENTS` below to rerun only a selected question/agent.

In [ ]:
import asyncio
from uuid import uuid4
from azure.ai.projects.aio import AIProjectClient as AsyncProjectClient
from azure.identity.aio import DefaultAzureCredential as AsyncDefaultAzureCredential
from azure.core.exceptions import AzureError
from agent_framework.exceptions import AgentException
from mcp.shared.exceptions import McpError
from openai import APITimeoutError, OpenAIError

AGENT_TIMEOUT_SECONDS = 180
DOC_ID = re.compile(r"DOC-[A-Z]+-[A-Z0-9]+(?:-\d+)?")
SUPERSEDED = set(register.loc[register.document_status == "Superseded", "document_id"])

async def ask_agent(name, question):
    t0 = time.perf_counter()
    async with asyncio.timeout(AGENT_TIMEOUT_SECONDS):
        async with AsyncDefaultAzureCredential() as async_credential, AsyncProjectClient(
            endpoint=FOUNDRY_PROJECT_ENDPOINT, credential=async_credential, allow_preview=True,
        ) as async_project:
            async with async_project.get_openai_client(
                agent_name=name, max_retries=0, timeout=AGENT_TIMEOUT_SECONDS,
                default_headers=COPILOT_HEADERS if name == HARNESS_AGENT else None,
            ) as client:
                conversation = await client.conversations.create()
                print(f"  {name}: conversation {conversation.id}", flush=True)
                response = await client.responses.create(conversation=conversation.id, input=question)
    calls = [item.name for item in response.output if item.type == "mcp_call"]
    errors = [f"{item.name}: {item.error}" for item in response.output if item.type == "mcp_call" and item.error]
    if response.error:
        errors.append(str(response.error))
    if response.incomplete_details:
        errors.append(str(response.incomplete_details))
    return {"answer": response.output_text, "tool calls": ", ".join(calls), "error": "\n".join(errors),
            "status": "tool_error" if errors and response.status == "completed" else response.status,
            "conversation id": conversation.id, "response id": response.id,
            "tokens": response.usage.total_tokens if response.usage else None,
            "seconds": round(time.perf_counter() - t0, 1)}

async def ask_hosted(question):
    t0 = time.perf_counter()
    async with asyncio.timeout(AGENT_TIMEOUT_SECONDS):
        result = await hosted.agent.run(question)
    calls = [c.name for m in result.messages for c in m.contents if c.type == "function_call"]
    errors = [str(c.exception) for m in result.messages for c in m.contents
              if c.type == "function_result" and c.exception]
    usage = result.usage_details or {}
    return {"answer": result.text, "tool calls": ", ".join(calls), "error": "\n".join(errors),
            "status": "tool_error" if errors else "completed", "tokens": usage.get("total_token_count"),
            "seconds": round(time.perf_counter() - t0, 1)}

SHOWCASE = ["Q05", "Q09", "Q11"]
SHOWCASE_AGENTS = ["prompt", "prompt + harness", "hosted"]
if AGENTS_LIVE:
    # CHARGEABLE: up to 9 agent runs with the default selections; no automatic SDK request retries.
    comparison_path = SAMPLE_OUT / "eval" / f"nb9-agent-comparison-{uuid4().hex}.jsonl"
    comparison_path.parent.mkdir(parents=True, exist_ok=True)
    print(f"Results journal: {comparison_path}", flush=True)
    agent_names = {"prompt": PROMPT_AGENT, "prompt + harness": HARNESS_AGENT}
    rows = []
    agent_results = pd.DataFrame()
    for qid in SHOWCASE:
        question = questions.loc[qid].question
        for label in SHOWCASE_AGENTS:
            row = {"question": qid, "agent": label, "status": "running", "answer": "", "error": "",
                   "tool calls": "", "tokens": None, "seconds": None,
                   "started UTC": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
            rows.append(row)
            with comparison_path.open("a", encoding="utf-8") as journal:
                journal.write(json.dumps(row, ensure_ascii=False) + "\n")
            print(f"START {qid} / {label} (deadline {AGENT_TIMEOUT_SECONDS}s)", flush=True)
            t0 = time.perf_counter()
            try:
                row.update(await ask_hosted(question) if label == "hosted"
                           else await ask_agent(agent_names[label], question))
            except (TimeoutError, APITimeoutError) as exc:
                row.update(status="timeout", error=f"{type(exc).__name__}: {exc or 'Client deadline exceeded'}")
            except (OpenAIError, AzureError, AgentException, McpError, httpx.HTTPError, ExceptionGroup) as exc:
                row.update(status="error", error=repr(exc))
            finally:
                if row["status"] == "running":
                    row.update(status="interrupted", error="Run interrupted before returning a result; see traceback.")
                row["seconds"] = round(time.perf_counter() - t0, 1)
                cited = set(DOC_ID.findall(row["answer"]))
                row.update(cited=", ".join(sorted(cited)), **{"superseded cited": bool(cited & SUPERSEDED)})
                with comparison_path.open("a", encoding="utf-8") as journal:
                    journal.write(json.dumps(row, ensure_ascii=False) + "\n")
                agent_results = pd.DataFrame(rows)
                print(f"END {qid} / {label}: {row['status']} after {row['seconds']}s", flush=True)
                display(pd.DataFrame([row]).drop(columns="answer"))
                if row["answer"]:
                    print(row["answer"], flush=True)

In [1]:
if AGENTS_LIVE:
    for qid in SHOWCASE:
        print(f"{qid} · {questions.loc[qid].question}\nExpected: {questions.loc[qid].expected_answer}\n")
        for _, row in agent_results[agent_results.question == qid].iterrows():
            print(f"[{row.agent}] {row.status}\n{row.answer}\n{row.error}\n")
        print("-" * 100)

NameError: name 'AGENTS_LIVE' is not defined

**What to look for**
- Tool calls: the prompt agents show `mcp_call` items through the toolbox; the hosted agent shows its function calls.
- Q09: the prompt agents depend on the instructions to ignore the superseded manual; the hosted agent can't retrieve
  it, because the filter is in its code.
- Q11: whether each agent asks the ontology first and searches the documents with the finding or asset it returned.

| Pattern | Who runs the loop | Who controls tool behaviour | Fits when |
|---|---|---|---|
| Prompt agent | Foundry Agent Service | Tool configuration in the toolbox | Tools and instructions are enough, and Foundry should operate everything |
| Prompt agent with the Copilot harness | Foundry-operated harness | Toolbox plus harness tool settings | Multi-step work with files or subagents, still without code to host |
| Hosted agent | The development team's code (Agent Framework) | Code: filters, ordering, post-processing | Behaviour must be enforced in code, or the framework and dependencies are the team's choice |

## 6 · Evaluations and tracing (preview)  *(deck section 5)*

One question set, three agents. The two prompt agents are **evaluation targets**: Foundry sends each question to the
agent and scores the responses it generates. The hosted agent runs in this notebook, so its answers are generated
here and uploaded as a **response dataset**. All three runs use the same evaluators:

| Evaluator | What it scores | Response it reads |
|---|---|---|
| Task adherence (preview) | Whether the agent followed its instructions and constraints | Prompt agents: output items with tool calls; hosted agent: final answer |
| Task completion (preview) | Whether the agent completed the request with a usable answer | Same as task adherence |
| Similarity | How close the answer is to the expected answer from the sample | Final answer text |

As per official documentation, the recommended primary measure for agent evaluation is a rubric evaluator generated
from the agent's context; add one once the questions are stable.

Sources: https://learn.microsoft.com/en-us/azure/foundry/observability/how-to/evaluate-agent ·
https://learn.microsoft.com/en-us/azure/foundry/observability/how-to/cloud-evaluation-targets ·
https://learn.microsoft.com/en-us/azure/foundry/concepts/evaluation-evaluators/agent-evaluators

### Evaluation dataset

Six questions by default: ontology (Q05, Q12), documents with a superseded trap (Q09, Q13), an inspector's wording
(Q17) and both sources (Q11). `EVAL_ALL = True` uses the 12 questions the three agents can answer; the semantic model,
row-level security, live stream and Work IQ questions belong to other steps.

In [ ]:
EVAL_ALL = False   # True: 12 questions, about 36 agent runs plus judge calls
EVAL_IDS = (["Q01", "Q02", "Q05", "Q06", "Q07", "Q09", "Q10", "Q11", "Q12", "Q13", "Q16", "Q17"] if EVAL_ALL
            else ["Q05", "Q09", "Q11", "Q12", "Q13", "Q17"])
eval_items = [{"id": qid, "query": questions.loc[qid].question, "ground_truth": questions.loc[qid].expected_answer}
              for qid in EVAL_IDS]
eval_path = SAMPLE_OUT / "eval" / "nb9-agent-questions.jsonl"
eval_path.write_text("\n".join(json.dumps(item, ensure_ascii=False) for item in eval_items), encoding="utf-8")
EVAL_VERSION = time.strftime("%Y%m%d%H%M%S")
if AGENTS_LIVE:
    eval_dataset = project.datasets.upload_file(name="games-ops-questions", version=EVAL_VERSION, file_path=str(eval_path))
    print(f"Dataset {eval_dataset.name} version {EVAL_VERSION}: {len(eval_items)} questions")

### Evaluation of the two prompt agents

One evaluation definition, two runs with the agents as targets. Runs are long-running; the results cell polls them.

In [ ]:
from azure.ai.projects.models import TestingCriterionAzureAIEvaluator
from openai.types.eval_create_params import DataSourceConfigCustom

def evaluators(response_items, response_text):
    judge = {"deployment_name": FOUNDRY_MODEL_NAME}
    return [
        TestingCriterionAzureAIEvaluator(type="azure_ai_evaluator", name="Task adherence", evaluator_name="builtin.task_adherence",
            initialization_parameters=judge, data_mapping={"query": "{{item.query}}", "response": response_items}),
        TestingCriterionAzureAIEvaluator(type="azure_ai_evaluator", name="Task completion", evaluator_name="builtin.task_completion",   # verify https://learn.microsoft.com/en-us/azure/foundry/concepts/evaluation-evaluators/agent-evaluators
            initialization_parameters=judge, data_mapping={"query": "{{item.query}}", "response": response_items}),
        TestingCriterionAzureAIEvaluator(type="azure_ai_evaluator", name="Similarity", evaluator_name="builtin.similarity",
            initialization_parameters=judge,
            data_mapping={"query": "{{item.query}}", "response": response_text, "ground_truth": "{{item.ground_truth}}"}),
    ]

item_schema = {"type": "object", "required": ["query", "ground_truth"],
               "properties": {"id": {"type": "string"}, "query": {"type": "string"}, "ground_truth": {"type": "string"}}}
eval_runs = {}
if AGENTS_LIVE:
    openai_client = project.get_openai_client()        # created after tracing was turned on
    target_eval = openai_client.evals.create(
        name="Games operations prompt agents",
        data_source_config=DataSourceConfigCustom(type="custom", item_schema=item_schema, include_sample_schema=True),
        testing_criteria=evaluators("{{sample.output_items}}", "{{sample.output_text}}"),
    )
    # CHARGEABLE: one agent run per question and agent, plus judge-model calls
    for label, agent_name in (("prompt", PROMPT_AGENT), ("prompt + harness", HARNESS_AGENT)):
        run = openai_client.evals.runs.create(eval_id=target_eval.id, name=agent_name, data_source={
            "type": "azure_ai_target_completions",
            "source": {"type": "file_id", "id": eval_dataset.id},
            "input_messages": {"type": "template", "template": [
                {"type": "message", "role": "user", "content": {"type": "input_text", "text": "{{item.query}}"}}]},
            "target": {"type": "azure_ai_agent", "name": agent_name},
        })
        eval_runs[label] = (target_eval.id, run.id)
        print(f"Run started for {agent_name}: {run.id}")

### Evaluation of the hosted agent

The hosted agent answers the same questions in this notebook. The answers are uploaded as a dataset and scored
with the same evaluators, mapped to the dataset's `response` field.

In [ ]:
if AGENTS_LIVE:
    # CHARGEABLE: one hosted-agent run per question, plus judge-model calls
    hosted_items = []
    for item in eval_items:
        result = await ask_hosted(item["query"])
        if result["status"] != "completed":
            raise RuntimeError(f"{item['id']}: hosted retrieval failed: {result['error']}")
        hosted_items.append({**item, "response": result["answer"]})
    hosted_path = SAMPLE_OUT / "eval" / "nb9-hosted-responses.jsonl"
    hosted_path.write_text("\n".join(json.dumps(item, ensure_ascii=False) for item in hosted_items), encoding="utf-8")
    hosted_dataset = project.datasets.upload_file(name="games-ops-hosted-responses", version=EVAL_VERSION,
                                                  file_path=str(hosted_path))
    hosted_schema = {**item_schema, "properties": {**item_schema["properties"], "response": {"type": "string"}}}
    hosted_eval = openai_client.evals.create(
        name="Games operations hosted agent",
        data_source_config=DataSourceConfigCustom(type="custom", item_schema=hosted_schema, include_sample_schema=False),
        testing_criteria=evaluators("{{item.response}}", "{{item.response}}"),
    )
    run = openai_client.evals.runs.create(eval_id=hosted_eval.id, name=HOSTED_AGENT, data_source={
        "type": "jsonl", "source": {"type": "file_id", "id": hosted_dataset.id}})   # verify https://learn.microsoft.com/en-us/azure/foundry/how-to/develop/cloud-evaluation
    eval_runs["hosted"] = (hosted_eval.id, run.id)
    print(f"Run started for {HOSTED_AGENT}: {run.id}")

### Results

Polls the three runs, then shows passed results per evaluator and agent, the score per question, and the report
link for each run in the Foundry portal.

In [ ]:
if AGENTS_LIVE:
    summary, scores = [], []
    for label, (eval_id, run_id) in eval_runs.items():
        while (run := openai_client.evals.runs.retrieve(run_id=run_id, eval_id=eval_id)).status not in ("completed", "failed", "canceled"):
            time.sleep(10)
        print(f"{label}: {run.status} · {run.report_url}")
        for criterion in run.per_testing_criteria_results or []:
            summary.append({"agent": label, "evaluator": criterion.testing_criteria,
                            "passed": f"{criterion.passed}/{criterion.passed + criterion.failed}"})
        for output in openai_client.evals.runs.output_items.list(run_id=run_id, eval_id=eval_id):   # verify https://learn.microsoft.com/en-us/azure/foundry/how-to/develop/cloud-evaluation
            for result in output.results:
                scores.append({"agent": label, "question": output.datasource_item.get("id"),
                               "evaluator": result.name, "score": result.score})
    display(pd.DataFrame(summary).pivot(index="evaluator", columns="agent", values="passed"))
    display(pd.DataFrame(scores).pivot_table(index=["question", "evaluator"], columns="agent", values="score"))

### Traces

Every agent run left spans in the project's Application Insights: server-side for the prompt agents, client-side for
the hosted agent. This Kusto Query Language (KQL) query summarises the last two hours per agent and operation (agent
invocations, model calls, tool calls), with durations and tokens. Allow 2 to 5 minutes for ingestion. The same
traces open per response in the Foundry portal, under each agent's **Traces** tab.

Source: https://learn.microsoft.com/en-us/azure/foundry/observability/how-to/trace-agent-setup

In [ ]:
from datetime import timedelta
from azure.monitor.query import LogsQueryClient

APP_INSIGHTS_RESOURCE_ID = env("APP_INSIGHTS_RESOURCE_ID")
if AGENTS_LIVE and APP_INSIGHTS_RESOURCE_ID.startswith("/subscriptions/") and "YOUR-" not in APP_INSIGHTS_RESOURCE_ID:
    kql = f"""
dependencies
| where timestamp > ago(2h)
| extend agent = tostring(customDimensions["gen_ai.agent.name"]), operation = tostring(customDimensions["gen_ai.operation.name"]),
         input_tokens = toint(customDimensions["gen_ai.usage.input_tokens"]), output_tokens = toint(customDimensions["gen_ai.usage.output_tokens"])
| where agent in ("{PROMPT_AGENT}", "{HARNESS_AGENT}", "{HOSTED_AGENT}")
| summarize spans = count(), avg_ms = round(avg(duration), 0), input_tokens = sum(input_tokens), output_tokens = sum(output_tokens) by agent, operation
| order by agent asc, operation asc"""
    result = LogsQueryClient(credential).query_resource(APP_INSIGHTS_RESOURCE_ID, kql, timespan=timedelta(hours=2))
    table = result.tables[0]
    display(pd.DataFrame(table.rows, columns=table.columns))
else:
    print("APP_INSIGHTS_RESOURCE_ID not set — trace query skipped; open each agent's Traces tab in the Foundry portal")

## Cleanup

The first cell deletes what steps 5 and 6 created: the two prompt agents, the toolbox, the three project connections
and the agent knowledge base. The second deletes the knowledge base, the three knowledge sources, the objects
generated by the indexed OneLake source and the step 3 pipeline (indexer, skillset, data source, index). Set
`CLEANUP = True` to run them; later steps of this notebook reuse these objects, so keep them until the end of the demo.

**Not deleted:** the evaluations, evaluation runs and datasets (kept for comparison in the portal), the Foundry User
role assignments of the two agents (remove them in the Azure portal; they stop applying once the agent identities are
deleted), the Azure AI Search service (billed per hour for Basic and higher tiers), the Foundry model
deployments, the Fabric workspace and its items (lakehouse, eventhouse, semantic model, ontology) and the Fabric
capacity, which bills while it runs.

In [ ]:
# DESTRUCTIVE: deletes the agents, toolbox, project connections and agent knowledge base created in step 5
CLEANUP = False
if AGENTS_LIVE and CLEANUP:
    token = credential.get_token("https://management.azure.com/.default").token
    for action, name in [(lambda n: project.agents.delete(agent_name=n), PROMPT_AGENT),   # verify https://learn.microsoft.com/en-us/python/api/azure-ai-projects/
                         (lambda n: project.agents.delete(agent_name=n), HARNESS_AGENT),
                         (lambda n: project.toolboxes.delete(n), TOOLBOX),
                         (lambda n: arm.delete(f"/{n}", headers={"Authorization": f"Bearer {token}"}).raise_for_status(), CONN_TOOLBOX),
                         (lambda n: arm.delete(f"/{n}", headers={"Authorization": f"Bearer {token}"}).raise_for_status(), CONN_ONTOLOGY),
                         (lambda n: arm.delete(f"/{n}", headers={"Authorization": f"Bearer {token}"}).raise_for_status(), CONN_DOCS),
                         (index_client.delete_knowledge_base, AGENT_KB)]:
        try:
            action(name)
            print("deleted", name)
        except Exception as e:
            print("skipped", name, "-", type(e).__name__)

In [ ]:
# DESTRUCTIVE: deletes the search objects created in steps 3 and 4
CLEANUP = False
if LIVE and CLEANUP:
    for action, name in [(index_client.delete_knowledge_base, KB),
                         (index_client.delete_knowledge_source, KS_DOCS),
                         (index_client.delete_knowledge_source, KS_ONELAKE),
                         (index_client.delete_knowledge_source, KS_ONTOLOGY),
                         (indexer_client.delete_indexer, f"{KS_ONELAKE}-indexer"),
                         (indexer_client.delete_skillset, f"{KS_ONELAKE}-skillset"),
                         (indexer_client.delete_data_source_connection, f"{KS_ONELAKE}-datasource"),
                         (index_client.delete_index, f"{KS_ONELAKE}-index"),
                         (indexer_client.delete_indexer, INDEXER), (indexer_client.delete_skillset, SKILLSET),
                         (indexer_client.delete_data_source_connection, DATASOURCE), (index_client.delete_index, INDEX)]:
        try:
            action(name)
            print("deleted", name)
        except Exception as e:
            print("skipped", name, "-", type(e).__name__)

## What you built

| Step | Built / shown | Azure resource touched |
|---|---|---|
| 0 | Settings, clients, sample register and questions | Foundry project (read) |
| 1 | Ontology MCP tool list and one question | Fabric ontology (read) |
| 2 | Fabric IQ MCP tool list and one discovery call | Fabric semantic models (read) |
| 3 | `games-docs` index, skillset, OneLake data source and indexer; register fields merged into chunks | Azure AI Search; Foundry resource (extraction, embeddings) |
| 4 | Three knowledge sources, the `games-kb` knowledge base and three comparisons | Azure AI Search; Foundry resource (chat model, embeddings); Fabric ontology (read) |
| 5 | `games-agent-kb`, three project connections, toolbox `games-ops-tools`, agents `games-ops-prompt` and `games-ops-harness` with Foundry User for their identities, hosted agent run in the notebook | Foundry project (connections, toolbox, agents, role assignments); Azure AI Search; Fabric ontology (read) |
| 6 | Two datasets, two evaluations with three runs, a trace summary | Foundry project (datasets, evaluations); Application Insights (read) |

## Sources and tested versions

Written against `azure-search-documents` 12.1.0b2 (REST API version 2026-08-01-preview), `mcp` 1.x
(`streamablehttp_client`, as in notebooks 1 and 8), `azure-ai-projects` 2.10.0 (steps 5 and 6), `agent-framework` 1.x
and `azure-mgmt-authorization` 4.x. The step 5 harness request also passes offline serialization checks with
`azure-ai-projects` 2.8.0 using the SDK JSON body and explicit preview header; the shared requirements are unchanged.
Steps 0 to 4 were run by the owner on
2026-11-10; steps 5 and 6 are not yet executed against Azure.

- https://learn.microsoft.com/en-us/fabric/iq/ontology/how-to-use-ontology-mcp-server
- https://learn.microsoft.com/en-us/fabric/iq/connectors/fabric-iq-mcp
- https://learn.microsoft.com/en-us/azure/search/search-how-to-index-onelake-files
- https://learn.microsoft.com/en-us/azure/search/cognitive-search-skill-content-understanding
- https://learn.microsoft.com/en-us/azure/search/search-how-to-semantic-chunking-content-understanding
- https://learn.microsoft.com/en-us/azure/search/cognitive-search-skill-azure-openai-embedding
- https://learn.microsoft.com/en-us/azure/search/search-how-to-define-index-projections
- https://learn.microsoft.com/en-us/azure/search/cognitive-search-attach-cognitive-services
- https://learn.microsoft.com/en-us/azure/search/agentic-retrieval-how-to-create-index
- https://learn.microsoft.com/en-us/azure/search/agentic-knowledge-source-how-to-search-index
- https://learn.microsoft.com/en-us/azure/search/agentic-knowledge-source-how-to-onelake
- https://learn.microsoft.com/en-us/azure/search/agentic-knowledge-source-how-to-fabric-ontology
- https://learn.microsoft.com/en-us/azure/search/agentic-retrieval-how-to-create-knowledge-base
- https://learn.microsoft.com/en-us/azure/search/agentic-retrieval-how-to-retrieve
- https://github.com/Azure/azure-sdk-for-python/blob/main/sdk/search/azure-search-documents/CHANGELOG.md
- https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/fabric-iq
- https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/toolbox
- https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/foundry-iq-connect
- https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/use-tools-skills-copilot-harness
- https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/use-toolbox-hosted-agent
- https://learn.microsoft.com/en-us/azure/foundry/observability/how-to/evaluate-agent
- https://learn.microsoft.com/en-us/azure/foundry/observability/how-to/cloud-evaluation-targets
- https://learn.microsoft.com/en-us/azure/foundry/concepts/evaluation-evaluators/agent-evaluators
- https://learn.microsoft.com/en-us/azure/foundry/observability/how-to/trace-agent-client-side
- https://github.com/Azure/azure-sdk-for-python/blob/main/sdk/ai/azure-ai-projects/CHANGELOG.md

## Before you build on this: re-check …
- **Fabric sign-in scopes** for the ontology MCP endpoint and Fabric IQ MCP (`# verify` in step 1): the
  documentation names only the delegated Power BI permissions for Fabric IQ MCP and shows interactive sign-in for
  the ontology endpoint.
- **Content Understanding with OneLake:** whether the OneLake indexer passes the file bytes to the skill; step 3
  offers `CHUNKER = "builtIn"` as an explicit alternative when it doesn't.
- **Preview surfaces:** the Fabric ontology knowledge source, answer synthesis and reasoning effort need REST API
  version 2026-08-01-preview and the pre-release client; class names and the `fabricOntology` parameter kind can
  change between betas.
- **Agent identity and connections (step 5):** the Power BI audience on the ontology connection, the
  `instance_identity` field on the agent details and the RemoteTool connection API version are `# verify` items.
- **Copilot harness preview (step 5b):** the toolset classes and per-tool settings, region and model availability.
- **Evaluation surfaces (step 6):** the task completion evaluator name, the `jsonl` data source for the hosted agent's
  responses and the output-item fields; whether an agent-target run passes a user identity to Fabric IQ.
- **Agent Framework (step 5c):** the observability import and the response fields read for tool calls and tokens.
- **`mcp` 2.0** removed `streamablehttp_client`; the kit caps `mcp` below 2 until notebooks 1, 8 and 9 move to
  `streamable_http_client`.